In [ ]:
# ============================================================
# RANKING MODEL WEIGHT OPTIMIZER
# Multi-year backtest to find robust factor weights
# ============================================================
#
# This notebook runs the ranking model across 5 lookback periods
# (1y, 2y, 3y, 4y, 5y) and finds weights that maximize average
# Spearman correlation across all periods â preventing overfitting
# to any single market regime.
#
# Run all cells. Takes ~15-20 minutes (fetches data for ~76 stocks
# Ã 5 periods = ~380 yfinance calls).
# ============================================================

import os, sys, importlib

# --- Mount Google Drive ---
from google.colab import drive
drive.mount('/content/drive')
sys.path.insert(0, '/content/drive/MyDrive/Colab/Stock_portfolio')

# --- Reload modules ---
import portfolio.ranking
import portfolio.validation
import portfolio.optimizer
importlib.reload(portfolio.ranking)
importlib.reload(portfolio.validation)
importlib.reload(portfolio.optimizer)

from portfolio.optimizer import run_multi_year_optimization, factor_correlations

# ============================================================
# RUN OPTIMIZATION
# ============================================================
# This collects data for all 5 periods, computes factor correlations,
# and runs grid search + coordinate descent to find optimal weights.

results = run_multi_year_optimization(
    periods_months=[12, 24, 36, 48, 60],
    step=0.05,  # 5% grid increments (use 0.10 for faster but coarser)
)

# ============================================================
# RESULTS SUMMARY
# ============================================================
print("\n" + "=" * 60)
print("COPY THESE WEIGHTS TO ranking.py STRATEGY_WEIGHTS")
print("=" * 60)

if "strategy_weights" in results:
    for strat, weights in results["strategy_weights"].items():
        print(f'\n    "{strat}": {{')
        for f, w in weights.items():
            print(f'        "{f}": {w:.2f},')
        print("    },")

print(f"\nAdjustment multiplier: {results['adj_multiplier']}")
print(f"(1.0 = current, 0.0 = no adjustments, 0.5 = half)")
